In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('../data/bitext_raw.csv')
df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


# ANALISIS EXPLORATORIO DE LOS DATOS (EDA)
**Fuente:** [Bitext Customer Support LLM Chatbot Training Dataset](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) (Hugging Face)                                  
Dataset público de Hugging Face con 26,872 conversaciones de soporte al cliente 
(instrucción del usuario → respuesta esperada), categorizadas en 11 categorías y 27 intents.

**Pregunta de negocio:** ¿qué categorías concentran mayor volumen de consultas, 
y qué implica eso para priorizar dónde invertir en automatización de soporte?

## Estadística descriptiva

Resumimos las variables del dataset: por un lado las numéricas derivadas 
(longitud de instrucción y de respuesta, en caracteres) con sus medidas de 
tendencia central y variabilidad; por otro, las variables categóricas 
(categoría, intención, flags) con su cardinalidad y valores más frecuentes.

In [7]:
resumen_numerico = df[['longitud_instruccion', 'longitud_respuesta']].describe().T
resumen_numerico.columns = ['conteo', 'media', 'desvio_std', 'minimo', 'q1_25%', 'mediana_50%', 'q3_75%', 'maximo']
resumen_numerico['rango'] = resumen_numerico['maximo'] - resumen_numerico['minimo']
resumen_numerico['coef_variacion'] = (resumen_numerico['desvio_std'] / resumen_numerico['media']).round(2)
resumen_numerico['asimetria'] = df[['longitud_instruccion', 'longitud_respuesta']].skew().values
resumen_numerico['curtosis'] = df[['longitud_instruccion', 'longitud_respuesta']].kurt().values

resumen_numerico.round(1)

,conteo,media,desvio_std,minimo,q1_25%,mediana_50%,q3_75%,maximo,rango,coef_variacion,asimetria,curtosis
longitud_instruccion,26872.0,46.9,10.9,6.0,40.0,48.0,55.0,92.0,86.0,0.2,-0.2,0.4
longitud_respuesta,26872.0,634.1,331.6,57.0,427.0,540.0,753.0,2472.0,2415.0,0.5,1.7,3.7


In [8]:
print(f"Filas duplicadas exactas: {df.duplicated().sum()}")
print(f"Instrucciones duplicadas: {df['instruction'].duplicated().sum()}")

Filas duplicadas exactas: 0
Instrucciones duplicadas: 2237


## Calidad de datos

Antes de resumir estadísticamente, verificamos si hay filas o instrucciones duplicadas.

No se encontraron filas completamente duplicadas. Sí hay 2,237 instrucciones con texto 
idéntico a otra fila (8.3% del total) — esperable dado que la columna `flags` genera 
variantes controladas de una misma instrucción base, y algunas de esas variantes 
pueden coincidir textualmente aunque provengan de generaciones distintas.

In [6]:
resumen_categorico = pd.DataFrame({
    'valores_unicos': [df['category'].nunique(), df['intent'].nunique(), df['flags'].nunique()],
    'valor_mas_frecuente': [df['category'].mode()[0], df['intent'].mode()[0], df['flags'].mode()[0]],
    'frecuencia_moda': [df['category'].value_counts().iloc[0], df['intent'].value_counts().iloc[0], df['flags'].value_counts().iloc[0]]
}, index=['category', 'intent', 'flags'])

resumen_categorico

,valores_unicos,valor_mas_frecuente,frecuencia_moda
category,11,ACCOUNT,5986
intent,27,check_invoice,1000
flags,394,BL,5212


In [9]:
correlacion = df[['longitud_instruccion', 'longitud_respuesta']].corr()
correlacion

,longitud_instruccion,longitud_respuesta
longitud_instruccion,1.000000,0.106039
longitud_respuesta,0.106039,1.000000


**Correlación entre longitud de instrucción y longitud de respuesta:** 0.11 — 
prácticamente nula. La extensión de lo que pregunta el usuario no predice la 
extensión de la respuesta; esta última depende del tipo de consulta (categoría/intent), 
no de cuánto escribió quien consulta.

## Distribución de consultas por categoría

¿Qué categorías concentran mayor volumen de consultas, y qué implica eso 
para priorizar dónde invertir en automatización de soporte?

In [4]:
category_counts = df['category'].value_counts().reset_index()
category_counts.columns = ['categoria', 'cantidad']

fig = px.bar(
    category_counts, 
    x='categoria', 
    y='cantidad',
    title='Distribución de consultas por categoría',
    labels={'categoria': 'Categoría', 'cantidad': 'Cantidad de consultas'}
)
fig.show()

## Longitud de respuesta por categoría

¿Todas las categorías generan respuestas de longitud similar, o hay categorías 
que requieren explicaciones más elaboradas que otras?

In [10]:
fig = px.box(
    df,
    x='category',
    y='longitud_respuesta',
    title='Distribución de longitud de respuesta por categoría',
    labels={'category': 'Categoría', 'longitud_respuesta': 'Longitud de respuesta (caracteres)'}
)
fig.show()

**Hallazgo:** REFUND muestra la mayor dispersión de todas las categorías 
(rango intercuartil de ~450 a ~1570 caracteres, alcanzando el máximo del dataset 
en 2472), sugiriendo que las respuestas de reembolso varían mucho según el caso 
particular. CANCEL, en el otro extremo, es la categoría más compacta y con 
respuestas más cortas — consistente con ser un proceso más estandarizado.

## Jerarquía de categorías e intenciones

Cada categoría se compone de una o más intenciones específicas. 
¿Cómo se distribuye el volumen dentro de esa jerarquía?

In [11]:
fig = px.sunburst(
    df,
    path=['category', 'intent'],
    title='Jerarquía de categorías e intenciones',
)
fig.show()

## Forma de la distribución: longitud de respuesta por categoría

El primer boxplot mostró cuartiles y outliers. Con un violin plot vemos 
además la forma completa de cada distribución — ¿son todas unimodales, 
o hay categorías con dos "picos" de longitud (respuestas cortas y largas 
conviviendo, sin un punto medio típico)?

In [19]:
fig = px.violin(
    df,
    x='category',
    y='longitud_respuesta',
    box=True,
    title='Forma de la distribución de longitud de respuesta por categoría',
    labels={'category': 'Categoría', 'longitud_respuesta': 'Longitud de respuesta (caracteres)'},
    color='category',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

## Longitud de instrucción vs. longitud de respuesta, por categoría

La correlación global entre ambas variables es casi nula (0.11). 
¿Se mantiene esa falta de relación dentro de cada categoría individual, 
o hay categorías donde sí existe un patrón que el análisis agregado esconde?

In [20]:
fig = px.scatter(
    df,
    x='longitud_instruccion',
    y='longitud_respuesta',
    color='category',
    opacity=0.4,
    title='Longitud de instrucción vs. longitud de respuesta, por categoría',
    labels={'longitud_instruccion': 'Longitud de instrucción (caracteres)', 'longitud_respuesta': 'Longitud de respuesta (caracteres)'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

## Mapa de calor: categoría × intención

Vista compacta de cómo se distribuye el volumen de consultas cruzando 
categoría e intención — permite identificar de un vistazo qué combinaciones 
concentran más casos.

In [22]:
cross = pd.crosstab(df['category'], df['intent'])

fig = px.imshow(
    cross,
    title='Mapa de calor: categoría × intención',
    labels=dict(x='Intención', y='Categoría', color='Cantidad'),
    aspect='auto',
    color_continuous_scale='Reds'
)
fig.show()

## Panel resumen: cuatro métricas clave en un solo vistazo

Combinamos volumen, longitud de respuesta, cantidad de intents y los intents 
más frecuentes en un único panel — una vista consolidada de la estructura del dataset.

In [25]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Volumen por categoría', 'Longitud de respuesta (mediana)', 
                     'Cantidad de intents por categoría', 'Volumen por intent (top 10)')
)

cat_counts = df['category'].value_counts()
fig.add_trace(go.Bar(x=cat_counts.index, y=cat_counts.values, name='Volumen'), row=1, col=1)

median_by_cat = df.groupby('category')['longitud_respuesta'].median().sort_values(ascending=False)
fig.add_trace(go.Bar(x=median_by_cat.index, y=median_by_cat.values, name='Mediana'), row=1, col=2)

intents_by_cat = df.groupby('category')['intent'].nunique().sort_values(ascending=False)
fig.add_trace(go.Bar(x=intents_by_cat.index, y=intents_by_cat.values, name='Intents'), row=2, col=1)

top_intents = df['intent'].value_counts().head(10)
fig.add_trace(go.Bar(x=top_intents.index, y=top_intents.values, name='Top intents'), row=2, col=2)

fig.update_layout(height=800, showlegend=False, title_text='Panel resumen: Bitext Customer Support Dataset')
fig.show()

## Comparación multidimensional por categoría

Normalizando volumen, longitud mediana de respuesta y cantidad de intents en 
una misma escala (0 a 1), podemos comparar el "perfil" de cada categoría en 
un solo gráfico — ¿qué categorías destacan en más de una dimensión a la vez?

In [26]:
resumen_radar = pd.DataFrame({
    'categoria': cat_counts.index,
    'volumen': cat_counts.values,
    'longitud_mediana': df.groupby('category')['longitud_respuesta'].median().reindex(cat_counts.index).values,
    'n_intents': df.groupby('category')['intent'].nunique().reindex(cat_counts.index).values
})

for col in ['volumen', 'longitud_mediana', 'n_intents']:
    resumen_radar[col + '_norm'] = resumen_radar[col] / resumen_radar[col].max()

fig = go.Figure()
for _, row in resumen_radar.iterrows():
    fig.add_trace(go.Scatterpolar(
        r=[row['volumen_norm'], row['longitud_mediana_norm'], row['n_intents_norm']],
        theta=['Volumen', 'Longitud mediana', 'N° intents'],
        fill='toself',
        name=row['categoria']
    ))

fig.update_layout(title='Comparación multidimensional por categoría (normalizado)', showlegend=True)
fig.show()

## Síntesis de hallazgos

- **ACCOUNT concentra el mayor volumen de consultas** (5,986 filas, 22.3% del total) 
  y también la mayor variedad de intenciones (6 de 27) — es tanto la categoría más 
  frecuente como la estructuralmente más compleja, lo que la convierte en la principal 
  candidata para priorizar inversión en automatización.

- **CANCEL y SUBSCRIPTION son las categorías más simples y de menor volumen** — 
  un solo intent cada una, y las respuestas más cortas del dataset (mediana de 249 
  y 530 caracteres respectivamente), consistente con ser procesos más estandarizados.

- **REFUND muestra la mayor variabilidad en longitud de respuesta** (desvío estándar 
  de 566, casi igual a su propia mediana de 615) — sugiere que los casos de reembolso 
  van desde resoluciones simples hasta explicaciones extensas, dependiendo del caso 
  particular.

- **No hay relación entre lo que pregunta el usuario y la extensión de la respuesta** 
  (correlación de 0.11) — la complejidad de la respuesta depende del tipo de consulta 
  (categoría/intención), no de cuánto escribió quien consulta. Un usuario puede hacer 
  una pregunta de 10 palabras y recibir una respuesta de 2,000 caracteres si el tema 
  lo amerita.

- **El dataset no tiene filas duplicadas**, aunque un 8.3% de las instrucciones 
  comparten texto idéntico con otra fila — esperable dado que la columna `flags` 
  genera variantes controladas (typos, coloquialismos) de instrucciones base, 
  y algunas de esas variantes coinciden textualmente.

**Implicancia de producto:** si se estuviera priorizando qué categorías automatizar 
primero en un sistema de soporte, ACCOUNT sería la prioridad natural por volumen y 
complejidad combinados. REFUND, pese a menor volumen, requeriría un diseño de 
respuestas más flexible dada su alta variabilidad — no alcanza con un template único.